# 01 — Data Compilation and EDA
**SPJIMR ANA526-PPM | TALA Multimodal AI Strategy | Day 1**

## Purpose
Load all datasets, validate schemas and provenance, and produce exploratory visualisations that frame the Day 2 modelling work.

## Evidence Discipline — Read Before Running

| Evidence type | Valid sources | Use for |
|--------------|---------------|---------|
| `official` | weartala.com, impact reports, press releases | Claim corpus, RAG |
| `customer_experience` | Trustpilot, Reddit, Google Reviews, press | Quality, fit, durability, returns |
| `press` / `community` | Good on You, Guardian, Reddit threads | Sustainability validation |
| `creator_strategy` | Instagram/TikTok creator posts | Creator mix, platform strategy |
| `brand_positioning` | Brand-owned social content | Campaign tone, hashtag analysis |
| `competitor_benchmark` | Competitor profile snapshots | Cross-brand comparison |

> **Rule:** Never use Instagram/TikTok engagement metrics as evidence for product quality or sustainability claims.

## Prerequisites
1. Copy templates from `data/raw/templates/` to `data/raw/`.
2. Populate at least `official_claims`, `customer_reviews`, and `creator_posts` for TALA.
3. Register every source in `docs/source_log_template.md`.
4. Select kernel **Python (TALA Multimodal AI)** before running.

## 0. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Project root so src/ is importable
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingestion import load_csv, load_brands_config, list_available_datasets
from src.validation import validate_and_report, validate_all_files, print_summary_report

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_colwidth', 120)

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
BRANDS = ['TALA', 'Adanola', 'Girlfriend Collective', 'Oner Active']

print('Project root:', PROJECT_ROOT)
print('Available datasets:', list_available_datasets())

## 1. Validate All Data Files

Run schema + provenance validation across every CSV in `data/raw/`.
Fix any FAIL errors before continuing.

In [ ]:
results = validate_all_files(data_dir=DATA_RAW, verbose=True)
print_summary_report(results)

## 2. Load Official Claims Corpus

**Source:** TALA website (sustainability page, responsibility page, About page, press interviews).  
**Evidence type:** `official`  
**File:** `data/raw/official_claims_tala.csv`

In [ ]:
try:
    claims = load_csv('official_claims_tala.csv')
    validate_and_report(claims, 'official_claims', 'TALA Official Claims')
    print(f'\nRows: {len(claims)} | Brands: {claims["brand"].unique().tolist()}')
    display(claims.head(3))
except FileNotFoundError:
    print('FILE NOT FOUND: copy official_claims_template.csv → official_claims_tala.csv and populate it.')

## 3. Load Customer Reviews

**Source:** Trustpilot (public reviews, no login required).  
**Evidence type:** `customer_experience`  
**File:** `data/raw/customer_reviews_tala.csv`  
> Strip usernames before saving. Star rating optional for Reddit rows.

In [ ]:
try:
    reviews = load_csv('customer_reviews_tala.csv')
    validate_and_report(reviews, 'customer_reviews', 'TALA Customer Reviews')
    print(f'\nRows: {len(reviews)} | Platforms: {reviews["platform"].value_counts().to_dict()}')
    display(reviews.head(3))
except FileNotFoundError:
    print('FILE NOT FOUND: copy customer_reviews_template.csv → customer_reviews_tala.csv and populate it.')

## 4. Load Press and Reddit Sources

**Source:** Reddit (r/femalefashionadvice, r/gymsnark), Good on You, press coverage.  
**Evidence type:** `press` or `community`  
**File:** `data/raw/press_reddit_sources_tala.csv`

In [ ]:
try:
    press = load_csv('press_reddit_sources_tala.csv')
    validate_and_report(press, 'press_reddit_sources', 'TALA Press & Reddit')
    print(f'\nRows: {len(press)} | Types: {press["source_type"].value_counts().to_dict()}')
    display(press.head(3))
except FileNotFoundError:
    print('FILE NOT FOUND: copy press_reddit_sources_template.csv and populate it.')

## 5. Load Creator Posts

**Source:** Instagram/TikTok/YouTube creator mentions (manually compiled).  
**Evidence type:** `creator_strategy`  
**File:** `data/raw/creator_posts_tala.csv`

In [ ]:
try:
    creators = load_csv('creator_posts_tala.csv')
    validate_and_report(creators, 'creator_posts', 'TALA Creator Posts')
    print(f'\nRows: {len(creators)} | Platforms: {creators["platform"].value_counts().to_dict()}')
    display(creators.head(3))
except FileNotFoundError:
    print('FILE NOT FOUND: copy creator_posts_template.csv and populate it.')

## 6. Load Competitor Platform Data

**Source:** Manual snapshots of brand profiles on Instagram, TikTok, YouTube.  
**Evidence type:** `competitor_benchmark`  
**File:** `data/raw/competitor_platforms.csv`

In [ ]:
try:
    competitors = load_csv('competitor_platforms.csv')
    validate_and_report(competitors, 'competitor_platforms', 'Competitor Platforms')
    print(f'\nRows: {len(competitors)} | Brands: {competitors["brand"].unique().tolist()}')
    display(competitors)
except FileNotFoundError:
    print('FILE NOT FOUND: copy competitor_platforms_template.csv and populate it.')

## 7. Row Count and Coverage Summary

Quick audit of how much data is in each dataset before analysis.

In [ ]:
# Build a coverage table from loaded DataFrames.
# Only include frames that were successfully loaded above.
coverage = {}
for label, df in [
    ('official_claims', claims if 'claims' in dir() else None),
    ('customer_reviews', reviews if 'reviews' in dir() else None),
    ('press_reddit', press if 'press' in dir() else None),
    ('creator_posts', creators if 'creators' in dir() else None),
    ('competitor_platforms', competitors if 'competitors' in dir() else None),
]:
    if df is not None:
        coverage[label] = {
            'rows': len(df),
            'brands': df['brand'].nunique() if 'brand' in df.columns else '?',
            'citation_ready': int(df['citation_ready'].fillna(False).sum()) if 'citation_ready' in df.columns else 0,
            'synthetic_rows': int(df['synthetic'].fillna(False).sum()) if 'synthetic' in df.columns else 0,
        }

coverage_df = pd.DataFrame(coverage).T
print('\n── Dataset Coverage ──')
display(coverage_df)

## 8. Required Field Completeness Check

In [ ]:
# For each loaded dataset, show % completeness of provenance fields.
PROVENANCE_FIELDS = ['source_url', 'source_platform', 'collection_date', 'collected_by', 'evidence_type']

for label, df in [
    ('official_claims', claims if 'claims' in dir() else None),
    ('customer_reviews', reviews if 'reviews' in dir() else None),
    ('creator_posts', creators if 'creators' in dir() else None),
]:
    if df is None or df.empty:
        continue
    pct = {f: f'{100 * df[f].notna().mean():.0f}%' if f in df.columns else 'MISSING' for f in PROVENANCE_FIELDS}
    print(f'\n{label}:')
    for field, val in pct.items():
        flag = '✓' if val == '100%' else ('⚠' if val != 'MISSING' else '✗')
        print(f'  {flag} {field}: {val}')

## 9. Summary by Brand / Platform / Evidence Type

Pivot to understand data coverage before analysis.

In [ ]:
# Combine all loaded datasets with an evidence_type column.
# Only include frames with rows.
frames = []
for df in [claims if 'claims' in dir() else None,
           reviews if 'reviews' in dir() else None,
           press if 'press' in dir() else None,
           creators if 'creators' in dir() else None]:
    if df is not None and not df.empty and 'evidence_type' in df.columns:
        frames.append(df[['brand', 'evidence_type', 'source_platform']].copy())

if frames:
    combined = pd.concat(frames, ignore_index=True)
    print('\nRow counts by brand and evidence_type:')
    display(combined.groupby(['brand', 'evidence_type']).size().unstack(fill_value=0))

    print('\nRow counts by source_platform:')
    display(combined['source_platform'].value_counts().to_frame('count'))
else:
    print('No data loaded yet. Populate templates and re-run.')

## 10. EDA — Customer Reviews

**Evidence type:** `customer_experience` only.  
These plots feed directly into the claim–experience divergence analysis in Notebook 03.

In [ ]:
if 'reviews' in dir() and not reviews.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Star rating distribution
    if 'star_rating' in reviews.columns and reviews['star_rating'].notna().any():
        reviews['star_rating'].value_counts().sort_index().plot(
            kind='bar', ax=axes[0], color=sns.color_palette('Set2')[0]
        )
        axes[0].set_title('Star Rating Distribution — TALA Reviews')
        axes[0].set_xlabel('Stars')
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=0)

    # Complaint type distribution
    if 'complaint_type' in reviews.columns and reviews['complaint_type'].notna().any():
        reviews['complaint_type'].value_counts().plot(
            kind='barh', ax=axes[1], color=sns.color_palette('Set2')[1]
        )
        axes[1].set_title('Complaint Types — TALA Reviews')
        axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'eda_reviews_tala.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved to outputs/figures/eda_reviews_tala.png')
else:
    print('No reviews data yet. Populate customer_reviews_tala.csv first.')

## 11. EDA — Official Claims Corpus

**Evidence type:** `official` only.

In [ ]:
if 'claims' in dir() and not claims.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Claims by category
    claims['claim_category'].value_counts().plot(
        kind='barh', ax=axes[0], color=sns.color_palette('Set2')[2]
    )
    axes[0].set_title('Official Claims by Category — TALA')
    axes[0].set_xlabel('Count')

    # Claims by source type
    claims['source_type'].value_counts().plot(
        kind='barh', ax=axes[1], color=sns.color_palette('Set2')[3]
    )
    axes[1].set_title('Claims by Source Type — TALA')
    axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'eda_claims_tala.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No claims data yet. Populate official_claims_tala.csv first.')

## 12. EDA — Creator Posts

**Evidence type:** `creator_strategy` only.

In [ ]:
if 'creators' in dir() and not creators.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Creator tier distribution
    tier_order = ['nano', 'micro', 'mid', 'macro', 'mega']
    tier_counts = creators['creator_tier'].value_counts().reindex(tier_order, fill_value=0)
    tier_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2')[4])
    axes[0].set_title('Creator Tier Distribution — TALA')
    axes[0].set_xlabel('Tier')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)

    # Platform split
    creators['platform'].value_counts().plot(
        kind='bar', ax=axes[1], color=sns.color_palette('Set2')[5]
    )
    axes[1].set_title('Creator Posts by Platform — TALA')
    axes[1].set_xlabel('Platform')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'eda_creators_tala.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No creator data yet. Populate creator_posts_tala.csv first.')

## 13. EDA — Competitor Platform Snapshot

**Evidence type:** `competitor_benchmark` only.

In [ ]:
if 'competitors' in dir() and not competitors.empty and 'followers' in competitors.columns:
    pivot = competitors.pivot_table(index='brand', columns='platform', values='followers', aggfunc='max')
    pivot.plot(kind='bar', figsize=(10, 5), colormap='Set2')
    plt.title('Follower Counts by Brand and Platform (snapshot)')
    plt.ylabel('Followers')
    plt.xlabel('')
    plt.xticks(rotation=15)
    plt.legend(title='Platform')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'eda_competitor_followers.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No competitor data yet. Populate competitor_platforms.csv first.')

## 14. Missing Data Audit

In [ ]:
for label, df in [
    ('official_claims', claims if 'claims' in dir() else None),
    ('customer_reviews', reviews if 'reviews' in dir() else None),
    ('creator_posts', creators if 'creators' in dir() else None),
]:
    if df is None or df.empty:
        continue
    null_pct = (df.isnull().mean() * 100).round(1)
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    if null_pct.empty:
        print(f'{label}: no nulls.')
    else:
        print(f'\n{label} — columns with nulls:')
        display(null_pct.to_frame('% null'))

## 15. Save Cleaned Interim Datasets

In [ ]:
from src.ingestion import save_csv

saved = []
for fname, df in [
    ('official_claims_tala_clean.csv', claims if 'claims' in dir() else None),
    ('customer_reviews_tala_clean.csv', reviews if 'reviews' in dir() else None),
    ('press_reddit_sources_tala_clean.csv', press if 'press' in dir() else None),
    ('creator_posts_tala_clean.csv', creators if 'creators' in dir() else None),
    ('competitor_platforms_clean.csv', competitors if 'competitors' in dir() else None),
]:
    if df is not None and not df.empty:
        path = save_csv(df, fname, subfolder='interim')
        saved.append(str(path))

if saved:
    print('\nSaved to data/interim/:')
    for s in saved:
        print(f'  {s}')
    print('\n✓ Notebook 01 complete. Proceed to Notebook 02.')
else:
    print('Nothing to save — populate data/raw/ templates first.')